# LangChain agent class definitions

This notebook defines all available LangChain agent classes. Import into an experiment notebook with:
```python
%run ./langchain_agents.ipynb
```

In [ ]:
import json
import httpx

from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

from playpen.agents import ClemAgent, ClemObservation

In [ ]:
from pathlib import Path
_SUBAGENT_PROMPT = Path("subagent_prompt.txt").read_text()

In [ ]:
class CoreToolsAgent(ClemAgent):
    """Agent with remember / recall / observe / get_observations tools."""

    def __init__(self, model: ChatOpenAI, thread_id: str = "default"):
        super().__init__()
        self.base_thread_id = thread_id
        self.episode = 0
        self.memory = InMemorySaver()
        self.model = model
        self.store = {}
        self._tool_observations = []

        tools = [
            self._remember_rules(),
            self._recall_rules(),
            self._observe_game(),
            self._get_game_observations(),
        ]

        system_prompt = """You are a professional game-playing agent. You have memory tools to help you play any game effectively.

  STRATEGY: follow this order every turn:

  TURN 1 (rules turn):
    1. Store the goal, required response format, and constraints using remember_rules().
    2. Give your first response in the required format.

  TURN 2+ (every subsequent turn):
    1. Call observe_game() to record the new information you just received.
    2. Call get_game_observations() to review what has already happened.
    3. Call recall_rules() to keep to the game rules.
    4. Give your response based on the full picture.

  Never skip steps 1-2 on turn 2+. Observations are required before acting."""

        self.agent = create_agent(
            model=self.model,
            tools=tools,
            checkpointer=self.memory,
            system_prompt=system_prompt,
        )

    def reset(self):
        super().reset()
        self.episode += 1
        self.store.clear()
        self._tool_observations.clear()

    def get_memory_snapshot(self) -> list:
        config = {"configurable": {"thread_id": f"{self.base_thread_id}_ep{self.episode}"}}
        state = self.memory.get(config)
        if state is None:
            return []
        return state.get("channel_values", {}).get("messages", [])

    def _remember_rules(self):
        store = self.store
        @tool
        def remember_rules(key: str, value: str) -> str:
            """
            Store any important information concerning the game rules.

            Examples:
                remember_rules("goal", "describe the target without forbidden words")
                remember_rules("format", "CLUE: <text>")
                remember_rules("target", "first grid")
                remember_rules("forbidden", "cat, dog, pet")
            """
            store[key] = value
            return f"Stored: {key} = {value}"
        return remember_rules

    def _recall_rules(self):
        store = self.store
        @tool
        def recall_rules(key: str = "") -> str:
            """
            Retrieve stored information about the game rules.

            Args:
                key: Specific key, or empty for everything
            """
            if not store:
                return "Memory empty."
            if key and key in store:
                return f"{key}: {store[key]}"
            return "\n".join(f"- {k}: {v}" for k, v in store.items())
        return recall_rules

    def _observe_game(self):
        observations = self._tool_observations
        @tool
        def observe_game(observation: str) -> str:
            """
            Note something important you noticed during the game.

            Examples:
                observe("Grid 1 has a red circle")
                observe("The clue mentions 'round shape'")
                observe("Player said 'no' to animal question")
            """
            observations.append(observation)
            return f"Noted: {observation}"
        return observe_game

    def _get_game_observations(self):
        observations = self._tool_observations
        @tool
        def get_game_observations() -> str:
            """Get all observations you've noted during the game."""
            if not observations:
                return "No observations yet."
            return "\n".join(f"{i+1}. {o}" for i, o in enumerate(observations))
        return get_game_observations

    def act(self, last: ClemObservation) -> str:
        result = self.agent.invoke(
            {"messages": [{"role": "user", "content": last.content}]},
            config={
                "configurable": {"thread_id": f"{self.base_thread_id}_ep{self.episode}"},
                "recursion_limit": 100,
            },
        )
        for msg in reversed(result["messages"]):
            if isinstance(msg, AIMessage) and msg.content:
                return msg.content
        return "(no response)"

In [ ]:
class TagExtractorAgent(ClemAgent):
    """Agent that uses an extract_tags subagent to parse game rules on first turn."""

    def __init__(self, model: ChatOpenAI, thread_id: str = "default"):
        super().__init__()
        self.model = model
        self.memory = InMemorySaver()
        self.base_thread_id = thread_id
        self.episode = 0

        @tool
        def extract_tags(initial_prompt: str) -> str:
            """Extract the response tags that are necessary for the player from the game rules."""
            subagent = create_agent(model=self.model, tools=[], system_prompt=_SUBAGENT_PROMPT)
            result = subagent.invoke({"messages": [{"role": "user", "content": initial_prompt}]})
            final = next(m for m in reversed(result["messages"]) if isinstance(m, AIMessage))
            return final.content

        self.agent = create_agent(
            model=self.model,
            tools=[extract_tags],
            checkpointer=self.memory,
            system_prompt=(
                "You're a professional agent game player. You're going to play a game. There is a helpful tool called extract_tags that identifies the required response format. On your first turn, call extract_tags and use the result to format all future responses."
            ),
        )

    def reset(self):
        super().reset()
        self.episode += 1

    def get_memory_snapshot(self) -> list:
        config = {"configurable": {"thread_id": f"{self.base_thread_id}_ep{self.episode}"}}
        state = self.memory.get(config)
        if state is None:
            return []
        return state.get("channel_values", {}).get("messages", [])

    def act(self, last: ClemObservation) -> str:
        result = self.agent.invoke(
            {"messages": [{"role": "user", "content": last.content}]},
            config={"configurable": {"thread_id": f"{self.base_thread_id}_ep{self.episode}"}},
        )
        for msg in reversed(result["messages"]):
            if isinstance(msg, AIMessage) and msg.content:
                return msg.content
        return "(no response)"

In [ ]:
class TwoToolsAgent(ClemAgent):
    """Agent with perception tools and a short-term memory component.
       As LLMs have no sensors, they might need some additional grounding. 
       While clembench provides an available action space, whereas the tools will provide an observation storage.
       It is expected that the agents will build upon this perception and take less "wrong" action,
       as they will have less outdated/hallucinated 'knowledge'.
    """

    def __init__(self, model: ChatOpenAI, thread_id: str = "default"):
        super().__init__()
        self.base_thread_id = thread_id
        self.episode = 0
        self.memory = InMemorySaver()
        self.model = model
        self.store = {}
        self._tool_observations = []

        tools = [
            self._observe_game(),
            self._get_game_observations(),
        ]

        system_prompt = """You are a professional game-playing agent. You have two memory tools to help you play any game effectively.

  STRATEGY: follow this order every turn:

  TURN 1 (You recieve the game rules and the general outline):
    Do not use any tools.

  TURN 2+:
    1. Call observe_game() to record something important you noticed during the game.
    2. Call get_game_observations() to review what has already happened.
    You MUST call observe_game() and get_game_observations() before you give the answer! Never skip steps 1-2 on turn 2+."""

        self.agent = create_agent(
            model=self.model,
            tools=tools,
            checkpointer=self.memory,
            system_prompt=system_prompt,
        )

    def reset(self):
        super().reset()
        self.episode += 1
        self.store.clear()
        self._tool_observations.clear()

    def get_memory_snapshot(self) -> list:
        config = {"configurable": {"thread_id": f"{self.base_thread_id}_ep{self.episode}"}}
        state = self.memory.get(config)
        if state is None:
            return []
        return state.get("channel_values", {}).get("messages", [])

    def _observe_game(self):
        observations = self._tool_observations
        @tool
        def observe_game(observation: str) -> str:
            """
            Note something important you noticed during the game.

            Examples:
                observe("Grid 1 has a red circle")
                observe("The clue 'round shape' given by me misleaded the other player")
                observe("Player said 'no' to animal question")
            """
            observations.append(observation)
            return f"Noted: {observation}"
        return observe_game

    def _get_game_observations(self):
        observations = self._tool_observations
        @tool
        def get_game_observations() -> str:
            """Get all observations you've noted during the game."""
            if not observations:
                return "No observations yet."
            return "\n".join(f"{i+1}. {o}" for i, o in enumerate(observations))
        return get_game_observations

    def act(self, last: ClemObservation) -> str:
        print("DEBUG:", self.history)
        result = self.agent.invoke(
            {"messages": [{"role": "user", "content": last.content}]},
            config={
                "configurable": {"thread_id": f"{self.base_thread_id}_ep{self.episode}"},
                "recursion_limit": 100,
            },
        )
        for msg in reversed(result["messages"]):
            if isinstance(msg, AIMessage) and msg.content:
                return msg.content
        return "(no response)"

In [ ]:
class LongTermPlanningAgent(ClemAgent):
    """Agent with long term memory for planning its actions."""

    def __init__(self, model: ChatOpenAI, thread_id: str = "default"):
        super().__init__()
        self.base_thread_id = thread_id
        self.episode = 0
        self.memory = InMemorySaver()
        self.model = model
        self._tool_observations = []

        tools = [
            self._write_trajectory(),
            self._get_written_trajectories(),
        ]

        system_prompt = """You're a professional game player with planning tools.

    STRATEGY:

    On the FIRST turn: 
    1. Call write_trajectory() to save what you are about to answer.
    
    At the START of every subsequent episode:
    1. Call get_written_trajectories() to recall lessons from past episodes.
    2. Apply that knowledge to play better.

    At the END of every subsequent episode (just before you make your final move):
    1. Call write_trajectory() to save what you learned from the past interaction (What have you attempted? Was it successful or not?).
    Make your move in the format defined by the game rules. Do NOT output anything else!
    """

        self.agent = create_agent(
            model=self.model,
            tools=tools,
            checkpointer=self.memory,
            system_prompt=system_prompt,
        )

    def _write_trajectory(self):
        observations = self._tool_observations
        @tool
        def write_trajectory(observation: str) -> str:
            """
            Promptly reason about your performance throughout the past rounds and make notes for yourself.
            Example: "tried Option 1, unsuccessful; game proceeds; need to observe the remaining options
            """
            observations.append(observation)
            return f"Noted: {observation}"
        return write_trajectory

    def _get_written_trajectories(self):
        observations = self._tool_observations
        @tool
        def get_written_trajectories() -> str:
            """Get all strategies and trajectories you've noted during the game."""
            if not observations:
                return "No observations yet."
            return "\n".join(f"{i+1}. {o}" for i, o in enumerate(observations[-10:]))
        return get_written_trajectories

    def reset(self):
        super().reset()
        self.episode += 1

    def get_memory_snapshot(self) -> list:
        """Return the current LangGraph message history for this episode (last 10 messages)."""
        config = {"configurable": {"thread_id": f"{self.base_thread_id}_ep{self.episode}"}}
        state = self.memory.get(config)
        if state is None:
            return []
        msgs = state.get("channel_values", {}).get("messages", [])
        return msgs[-10:]

    def get_longterm_snapshot(self) -> dict:
        """Return the long-term memory state (all observations, not wiped between episodes)."""
        return {
            "observations": list(self._tool_observations),
        }

    def act(self, last: ClemObservation) -> str:
        memory_summary = ""
        if self._tool_observations:
            memory_summary = "STRATEGIES AND OBSERVATIONS FROM PAST EPISODES:\n"
            memory_summary += "Past observations:\n" + "\n".join(f"- {o}" for o in self._tool_observations[-10:])
        result = self.agent.invoke(
            {"messages": [{"role": "user", "content": last.content + "\n\n" + memory_summary}]},
            config={
                "configurable": {"thread_id": f"{self.base_thread_id}_ep{self.episode}"},
                "recursion_limit": 100,
            },
        )
        for msg in reversed(result["messages"]):
            if isinstance(msg, AIMessage) and msg.content:
                return msg.content
                
        return "(no response)"